#### Working with LinkedIN URLs of Key People

In [16]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from urllib.parse import urlparse, urlunparse
from rapidfuzz import fuzz
import re

Metadata=pd.read_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\Hoosier_WebScraping_Output-Metadata_All_Pages.xlsx")
print(Metadata.shape)
Metadata = Metadata.loc[(Metadata["Consider for Detailed Scraping"] == "Y") & (Metadata["Is in Indiana State?"] == "Yes"), ["Employer Name", "City"]].drop_duplicates()
Metadata['Employer Name'] = Metadata['Employer Name'].str.replace(r"[#']", '', regex=True)
Metadata_scraped_index = Metadata[Metadata['Employer Name']=="Kendall Pool Svc"].index[0]
Metadata = Metadata.loc[Metadata_scraped_index+1:]
print(Metadata.shape)

def clean_company_name(name):
    name = name.lower()
    name = re.sub(r'\b(company|co|ltd|limited|inc|corp|corporation|pvt|private|llc|llp|plc|gmbh|s\.a\.|s\.r\.l\.|bv|ag)\b', '', name)
    name = re.sub(r'[^\w\s]', '', name)  
    name = re.sub(r'\s+', ' ', name).strip()
    return name

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\IlaBarshilia\\OneDrive - OIA Global\\VS_Code\\Hoosier_WebScraping_Output-Metadata_All_Pages.xlsx'

In [11]:
Metadata.head()

,Employer Name,City
110041,Jehovahs Witnesses,Charlestown
110050,Jehovahs Witnesses,Indianapolis
110061,Jehovahs Witnesses,Aurora
110067,Jehovahs Witnesses,Bloomington
110068,Jehovahs Witnesses,Bluffton


In [12]:
# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# Step 1: Open Capital IQ login page
driver.get("https://www.capitaliq.spglobal.com")
logging.info("Opened Capital IQ login page.")

# Step 2: Enter email
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
logging.info("Email entered.")

# Step 3: Click "Next"
next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
next_button.click()
logging.info("Email submitted.")

# Step 4: Enter password
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)

# Step 5: Click "Sign In"
sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
sign_in_button.click()
logging.info("Password submitted.")

# Step 6: Wait for login to complete
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
logging.info("Login successful and page loaded.")

# Step 7: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
except:
    pass

# Step 8: Navigate directly to the search results URL
all_company_data = []

for index, row in Metadata.iterrows():
    search_term = row['Employer Name']
    search_city = row['City']
    
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?vertical=private_companies_mi-gss&q={search_term.replace(' ', '%20')}"
    driver.get(search_url)
    logging.info(f"Processing: '{search_term}' Company in '{search_city}' City")
    try:
        clear_all_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Clear All']")))
        clear_all_button.click()
    except:
        pass

    dropdown_toggle = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@title='Select Geography']")))
    dropdown_toggle.click()
    search_input = wait.until(EC.presence_of_element_located((By.XPATH, "//input[@placeholder='Search Geography']")))
    search_input.send_keys("Indiana")
    # Wait for the "USA" option to appear and click it
    usa_option = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@title='Indiana' and contains(@class, 'css-b0wbzt')]")))
    usa_option.click()

    # Step 8: Check if search results loaded
    search_results_found = True
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
    except:
        search_results_found = False
        logging.warning(f"No Search results for {search_term}, continuing to next company")
        company_data = {
            "Employer Name": search_term,
            "City": search_city,
            "NAICS Code": None,
            "Industry": None,
            "Description": None,
            "Website URL": None,
            "Headquarters": None,
            "Current_Headcount": None,
            "Percent_Change_in_HC": None,
            "Change_in_HC": None,
            "CEO Name": None,
            "CEO Profile Link": None,
            "LinkedIN URL": None,
            "Key People": None
        }
        all_company_data.append(company_data)
        
    if not search_results_found:
        continue

    # Step 8.1: Handle cookie popup
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
        cookie_button.click()
    except:
        pass

    # Step 9: Change results per page to 50
    try:
        show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
        driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
        time.sleep(1)

        dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
        driver.execute_script("arguments[0].click();", dropdown_trigger)
        time.sleep(1)

        options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
        for option in options:
            if option.is_displayed():
                driver.execute_script("arguments[0].click();", option)
                # logging.info("✅ Changed results per page to 50")
                time.sleep(3)
                break
    except Exception as e:
        pass

    # Step 10: Find matching company by city
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')))
        company_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')
        match_found = False
        
        for link in company_links:
            try:
                found_city_text = link.find_element(By.XPATH, './ancestor::div[3]').text
                found_company_text = link.text.strip()
                cleaned_found_company = clean_company_name(found_company_text)
                if search_city.lower() in found_city_text.lower():
                    cleaned_search_company = clean_company_name(search_term)
                    similarity = fuzz.token_sort_ratio(cleaned_search_company, cleaned_found_company)
                    if similarity > 80:  # Adjust threshold as needed
                        logging.info(f"✅ Match found for: {link.text.strip()} | City: {search_city} | Similarity Score: {similarity}")
                        driver.execute_script("arguments[0].click();", link)
                        match_found = True

                        # Step 11: Initialize company_data for matched company
                        company_data = {
                            "Employer Name": search_term,
                            "City": search_city,
                            "NAICS Code": None,
                            "Industry": None,
                            "Description": None,
                            "Website URL": None,
                            "Headquarters": None,
                            "Current_Headcount": None,
                            "Percent_Change_in_HC": None,
                            "Change_in_HC": None,
                            "CEO Name": None,
                            "CEO Profile Link": None,
                            "LinkedIN URL": None,
                            "Key People": None
                        }
                        # Continue with data extraction steps...

                        # 1. NAICS Code
                        try:
                            naics_element = wait.until(EC.presence_of_element_located(
                                (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
                            company_data["NAICS Code"] = naics_element.text.strip()
                        except Exception as e:
                            pass

                        # 2. Industry
                        try:
                            industry_element = wait.until(EC.presence_of_element_located(
                                (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
                            company_data["Industry"] = industry_element.text.strip()
                        except Exception as e:
                            pass

                        # 3. Description
                        try:
                            # Try clicking "VIEW ALL" if it exists
                            try:
                                view_all_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
                                view_all_button.click()
                                time.sleep(1)
                            except:
                                pass

                            # Now extract the description
                            description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
                            description = description_element.text.strip()

                            # Clean up if "VIEW LESS" is included
                            if "VIEW LESS" in description:
                                description = description.split("VIEW LESS")[0].strip()
                            
                            # Remove "As of Date" line if present
                            if "As of Date:" in description:
                                description = description.split("As of Date:")[0].strip()

                            company_data["Description"] = description

                        except Exception as e:
                            company_data["Description"] = None

                        # try:
                        #     view_all_button = wait.until(EC.element_to_be_clickable(
                        #         (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
                        #     view_all_button.click()
                        #     time.sleep(1)
                        #     description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
                        #     description = description_element.text.strip()
                        #     if "VIEW LESS" in description:
                        #         description = description.split("VIEW LESS")[0].strip()
                        #     company_data["Description"] = description
                        # except Exception as e:
                        #     print("Description not found!")

                        # 4. Website URL
                        try:
                            cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
                            company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
                        except Exception as e:
                            pass

                        # 5. Headquarters
                        try:
                            block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
                            hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
                            lines = hq_element.text.splitlines()
                            company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
                        except Exception as e:
                            pass

                        # 6. Current Headcount
                        try:
                            headcount_element = wait.until(EC.presence_of_element_located((
                                By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
                            company_data["Current_Headcount"] = headcount_element.text.strip()
                        except Exception as e:
                            pass

                        # 7. Percent Change in Headcount
                        try:
                            percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
                            company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
                        except Exception as e:
                            pass

                        # 8. Change in Headcount Direction
                        try:
                            change_type = wait.until(EC.presence_of_element_located((By.XPATH,
                                '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
                            icon_html = change_type.get_attribute('outerHTML')
                            if 'caret-up' in icon_html:
                                company_data["Change_in_HC"] = 1
                            elif 'caret-down' in icon_html:
                                company_data["Change_in_HC"] = -1
                            else:
                                company_data["Change_in_HC"] = 0
                        except Exception as e:
                            pass

                        # 9. CEO Name and Profile Link
                        try:
                            ceo_element = wait.until(EC.presence_of_element_located((
                                By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
                            company_data["CEO Name"] = ceo_element.text
                            company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
                        except Exception as e:
                            pass

                        # 11. Officers
                        try:
                            # Step 1: Click the "More" link under Officers & Directors
                            more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
                            driver.execute_script("arguments[0].click();", more_button)
                            time.sleep(5)

                            # Step 2: Wait for the table to appear
                            wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'ag-root-wrapper')]")))
                            # Locate the grid rows
                            rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'ag-center-cols-container')]/div[contains(@class, 'ag-row')]")

                            merged_data = []
                            for i, row in enumerate(rows):
                                try:
                                    # Each cell is a div with class ag-cell
                                    cells = row.find_elements(By.XPATH, ".//div[contains(@class, 'ag-cell')]")
                                    if len(cells) >= 2:
                                        name = cells[0].text.strip()
                                        title = cells[1].text.strip()
                                        merged_data.append({
                                            "Name": name if name else "[No Name]",
                                            "Title": title if title else "[No Title]"
                                        })

                                except Exception as e:
                                    pass

                            # final_output = '; '.join(merged_data)
                            company_data["Key People"] = merged_data
                            # Navigate to the company's LinkedIn "people" page
                        except Exception as e:
                            company_data["Key People"] = None

                        
                        if company_data.get("Key People") is not None:
                            # Step 12: Get LinkedIN URLs of Key People
                            try:
                                wait = WebDriverWait(driver, 10)
                                # Find the span with the text and get its parent anchor tag
                                linkedin_element = driver.find_element(By.CSS_SELECTOR, 'a.spg-link[aria-label="View LinkedIn Professional"]')
                                linkedin_url = linkedin_element.get_attribute('href')
                                parsed_url = urlparse(linkedin_url)
                                clean_path = parsed_url.path.replace('//', '/').rstrip('/')
                                clean_url = urlunparse(parsed_url._replace(path=clean_path, query=''))
                                company_data["LinkedIN URL"] = clean_url
                                driver.execute_script("window.open(arguments[0]);", clean_url)
                                driver.switch_to.window(driver.window_handles[-1])
                            except Exception as e:
                                company_data["LinkedIN URL"] = None  
                                
                            # After switching to the LinkedIn tab
                            # Step 13: Log in to LinkedIn
                            try:
                                # Wait for login page to load
                                wait.until(EC.presence_of_element_located((By.ID, 'username')))

                                # Enter credentials
                                username_input = driver.find_element(By.ID, 'username')
                                password_input = driver.find_element(By.ID, 'password')

                                username_input.send_keys("ilabarshilia@gmail.com")  # Replace with your LinkedIn email
                                password_input.send_keys("Iambrilliant20")           # Replace with your LinkedIn password

                                # Click login
                                login_button = driver.find_element(By.XPATH, '//button[@type="submit"]')
                                login_button.click()

                                # Wait for login to complete
                                wait.until(EC.presence_of_element_located((By.ID, 'global-nav-search')))
                            except Exception as e:
                                print("Login failed")

                            # Step 2: Search each person and extract profile URL
                            linkedin_profiles = []
                            for person in company_data["Key People"]:
                                name = person.get("Name")
                                try:
                                    # Locate and use the LinkedIn search bar
                                    driver.get(linkedin_url)
                                    search_box = wait.until(EC.presence_of_element_located((By.ID, 'people-search-keywords')))
                                    search_box.clear()
                                    search_box.send_keys(name)
                                    search_box.send_keys(Keys.RETURN)
                                    time.sleep(5)

                                    # Wait for and extract the first profile link
                                    profile_link = wait.until(EC.presence_of_element_located((By.XPATH, '//a[contains(@href, "/in/")]')))
                                    person["LinkedIN Profile"] = profile_link.get_attribute('href')
                                except Exception as e:
                                    linkedin_profiles[name] = None
                                
                        all_company_data.append(company_data)
                        break
            except Exception as e:
                company_data = {
                "Employer Name": search_term,
                "City": search_city,
                "NAICS Code": None,
                "Industry": None,
                "Description": None,
                "Website URL": None,
                "Headquarters": None,
                "Current_Headcount": None,
                "Percent_Change_in_HC": None,
                "Change_in_HC": None,
                "CEO Name": None,
                "CEO Profile Link": None,
                "LinkedIN URL": None,
                "Key People": None
            }
                all_company_data.append(company_data)
                print(f"⚠️ Error processing link")


        if not match_found:
            logging.info(f"❗No matching company found for search_term: {search_term} with search_city:{search_city}")
            company_data = {
                "Employer Name": search_term,
                "City": search_city,
                "NAICS Code": None,
                "Industry": None,
                "Description": None,
                "Website URL": None,
                "Headquarters": None,
                "Current_Headcount": None,
                "Percent_Change_in_HC": None,
                "Change_in_HC": None,
                "CEO Name": None,
                "CEO Profile Link": None,
                "LinkedIN URL": None,
                "Key People": None
            }
            all_company_data.append(company_data)
            continue

    except Exception as e:
        print(f"❌ Error locating company links")
        company_data = {
                "Employer Name": search_term,
                "City": search_city,
                "NAICS Code": None,
                "Industry": None,
                "Description": None,
                "Website URL": None,
                "Headquarters": None,
                "Current_Headcount": None,
                "Percent_Change_in_HC": None,
                "Change_in_HC": None,
                "CEO Name": None,
                "CEO Profile Link": None,
                "LinkedIN URL": None,
                "Key People": None
            }
        all_company_data.append(company_data)
        continue

driver.quit()

2025-08-13 13:53:49 - INFO - Opened Capital IQ login page.
2025-08-13 13:53:57 - INFO - Email entered.
2025-08-13 13:53:57 - INFO - Email submitted.
2025-08-13 13:54:00 - INFO - Password submitted.
2025-08-13 13:54:00 - INFO - Login successful and page loaded.
2025-08-13 13:54:21 - INFO - Processing: 'Jehovahs Witnesses' Company in 'Charlestown' City


❌ Error locating company links


2025-08-13 13:55:09 - INFO - Processing: 'Jehovahs Witnesses' Company in 'Indianapolis' City


❌ Error locating company links


2025-08-13 13:55:53 - INFO - Processing: 'Jehovahs Witnesses' Company in 'Aurora' City


...

Login failed
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link
⚠️ Error processing link


2025-08-13 17:07:31 - INFO - Processing: 'Kelvin Handyman Svc' Company in 'Indianapolis' City
2025-08-13 17:07:42 - WARNING - No Search results for Kelvin Handyman Svc, continuing to next company
2025-08-13 17:07:42 - INFO - Processing: 'Kelvion Inc' Company in 'Indianapolis' City
2025-08-13 17:07:59 - INFO - ❗No matching company found for search_term: Kelvion Inc with search_city:Indianapolis
2025-08-13 17:07:59 - INFO - Processing: 'Kemna Restoration' Company in 'Indianapolis' City
2025-08-13 17:08:15 - INFO - ❗No matching company found for search_term: Kemna Restoration with search_city:Indianapolis
2025-08-13 17:08:16 - INFO - Processing: 'Kemna Restoration & Constr' Company in 'Indianapolis' City
2025-08-13 17:08:32 - INFO - ✅ Match found for: Kemna Restoration & Construction, Inc. | City: Indianapolis | Similarity Score: 88.88888888888889
2025-08-13 17:09:15 - INFO - Processing: 'Kemp Inc' Company in 'Cannelburg' City
2025-08-13 17:09:32 - INFO - ❗No matching company found for se

❌ Error locating company links


2025-08-13 17:10:23 - INFO - Processing: 'Ken Cut Lawn Svc Inc' Company in 'Indianapolis' City
2025-08-13 17:10:40 - INFO - ❗No matching company found for search_term: Ken Cut Lawn Svc Inc with search_city:Indianapolis
2025-08-13 17:10:44 - INFO - Processing: 'Ken Gates Ministries Inc' Company in 'Indianapolis' City
2025-08-13 17:10:55 - WARNING - No Search results for Ken Gates Ministries Inc, continuing to next company
2025-08-13 17:10:56 - INFO - Processing: 'Ken Kleen Inc' Company in 'Indianapolis' City
2025-08-13 17:11:07 - WARNING - No Search results for Ken Kleen Inc, continuing to next company
2025-08-13 17:11:07 - INFO - Processing: 'Ken Maddox Htg & Air Cond Inc' Company in 'Indianapolis' City
2025-08-13 17:11:19 - WARNING - No Search results for Ken Maddox Htg & Air Cond Inc, continuing to next company
2025-08-13 17:11:19 - INFO - Processing: 'Ken Skinner Construction' Company in 'Bloomfield' City
2025-08-13 17:11:30 - WARNING - No Search results for Ken Skinner Construction

KeyboardInterrupt: 

In [13]:
# Create a DataFrame
import pandas as pd
df2 = pd.DataFrame(all_company_data)

# Step 1: Explode the 'Key People' column to separate rows
company_data_exploded = df2.explode('Key People')

# Step 2: Extract 'Name' and 'Title' from each dictionary
company_data_exploded['Executive Name'] = company_data_exploded['Key People'].apply(lambda x: x['Name'] if isinstance(x, dict) and 'Name' in x else None)
company_data_exploded['Executive Title'] = company_data_exploded['Key People'].apply(lambda x: x['Title'] if isinstance(x, dict) and 'Title' in x else None)
company_data_exploded['LinkedIN Profile'] = company_data_exploded['Key People'].apply(lambda x: x['LinkedIN Profile'] if isinstance(x, dict) and 'LinkedIN Profile' in x else None)

# Step 3: Drop the original 'Key People' column
company_data_exploded = company_data_exploded.drop(columns=['Key People'])
# Step 4: Map LinkedIn profiles to each executive
# company_data_exploded['LinkedIN Profiles'] = company_data_exploded['Executive Name'].map(linkedin_profiles)
# new_col_order = ["Employer Name", "City", "NAICS Code", "Industry", "Description", "Website URL", "Headquarters",
#                  "Current_Headcount", "Percent_Change_in_HC", "Change_in_HC", "CEO Name","CEO Profile Link", "LinkedIN URL", 
#                  "Executive Name", "Executive Title", "LinkedIN Profiles"]
# company_data_exploded = company_data_exploded[new_col_order]

In [14]:
company_data_exploded

,Employer Name,City,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,CEO Name,CEO Profile Link,LinkedIN URL,Executive Name,Executive Title,LinkedIN Profile
0,Jehovahs Witnesses,Charlestown,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
1,Jehovahs Witnesses,Indianapolis,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
2,Jehovahs Witnesses,Aurora,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
3,Jehovahs Witnesses,Bloomington,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
4,Jehovahs Witnesses,Bluffton,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,Kendall Electric Inc,Angola,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
549,Kendall Electric Inc,Auburn,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
550,Kendall Inventory Svc Llc,Indianapolis,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None
551,Kendall Pest Control,Indianapolis,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None


In [15]:
company_data_exploded.to_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\CapIQ_Web_Scraping_Output_110042-115149_Companies.xlsx", index=False)

In [15]:
# Create a DataFrame
df2 = pd.DataFrame(all_company_data)

# Step 1: Explode the 'Key People' column to separate rows
company_data_exploded = df2.explode('Key People')

# Step 2: Extract 'Name' and 'Title' from each dictionary
company_data_exploded['Executive Name'] = company_data_exploded['Key People'].apply(lambda x: x['Name'] if isinstance(x, dict) and 'Name' in x else None)
company_data_exploded['Executive Title'] = company_data_exploded['Key People'].apply(lambda x: x['Title'] if isinstance(x, dict) and 'Title' in x else None)

# Step 3: Drop the original 'Key People' column
company_data_exploded = company_data_exploded.drop(columns=['Key People'])
company_data_exploded
# Step 4: Map LinkedIn profiles to each executive
# company_data_exploded['LinkedIN Profiles'] = company_data_exploded['Executive Name'].map(linkedin_profiles)
# new_col_order = ["Employer Name", "City", "NAICS Code", "Industry", "Description", "Website URL", "Headquarters",
#                  "Current_Headcount", "Percent_Change_in_HC", "Change_in_HC", "CEO Name","CEO Profile Link", "LinkedIN URL", 
#                  "Executive Name", "Executive Title", "LinkedIN Profiles"]
# company_data_exploded = company_data_exploded[new_col_order]

,Employer Name,City,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,CEO Name,CEO Profile Link,LinkedIN URL,LinkedIN Profiles,Executive Name,Executive Title
0,Polygon Co,Walkerton,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",74,5,-1,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,https://www.linkedin.com/company/polygon-compo...,{'Bree Katulak': 'https://www.linkedin.com/in/...,Bree Katulak,President & CEO
0,Polygon Co,Walkerton,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",74,5,-1,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,https://www.linkedin.com/company/polygon-compo...,{'Bree Katulak': 'https://www.linkedin.com/in/...,Jim Hoffman,Vice President of Finance & IT
0,Polygon Co,Walkerton,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",74,5,-1,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,https://www.linkedin.com/company/polygon-compo...,{'Bree Katulak': 'https://www.linkedin.com/in/...,Ben Fouch,Vice President of Strategy & Operations
0,Polygon Co,Walkerton,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",74,5,-1,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,https://www.linkedin.com/company/polygon-compo...,{'Bree Katulak': 'https://www.linkedin.com/in/...,Doug Drummond,Vice President of Sales & Marketing


In [17]:
df2["Key People"]

0    [{'Name': 'Bree Katulak', 'Title': 'President ...
Name: Key People, dtype: object

In [15]:
df2 = pd.DataFrame(all_company_data)
df2.to_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\Capital_IQ_WebScraping_100Companies_Consolidated.xlsx", index=False)

In [14]:
# Step 1: Explode the 'Key People' column to separate rows
company_data_exploded = df2.explode('Key People')

# Step 2: Extract 'Name' and 'Title' from each dictionary
company_data_exploded['Executive Name'] = company_data_exploded['Key People'].apply(lambda x: x['Name'] if isinstance(x, dict) and 'Name' in x else None)
company_data_exploded['Executive Title'] = company_data_exploded['Key People'].apply(lambda x: x['Title'] if isinstance(x, dict) and 'Title' in x else None)

# Step 3: Drop the original 'Key People' column
company_data_exploded = company_data_exploded.drop(columns=['Key People'])
# Step 4: Map LinkedIn profiles to each executive
company_data_exploded['LinkedIN Profiles'] = company_data_exploded['Executive Name'].map(linkedin_profiles)
new_col_order = ["Employer Name", "City", "NAICS Code", "Industry", "Description", "Website URL", "Headquarters",
                 "Current_Headcount", "Percent_Change_in_HC", "Change_in_HC", "CEO Name","CEO Profile Link", "LinkedIN URL", 
                 "Executive Name", "Executive Title", "LinkedIN Profiles"]
company_data_exploded = company_data_exploded[new_col_order]
company_data_exploded.to_excel(r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\Capital_IQ_WebScraping_100Companies_Exploded.xlsx", index=False)